In [1]:
import numpy as np
import h5py
import yaml
import os

import scipy.io
from scipy import signal

import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F
import torch.optim as optim

import utils
import random
from IPython.display import Audio

import librosa.display
import matplotlib.pyplot as plt
%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [5]:
path = 'data'
data = h5py.File(os.path.join(path, 'madeeg_preprocessed.hdf5'), 'r')
stream = open(os.path.join(path, 'madeeg_preprocessed.yaml'), 'r')
metadata = yaml.load(stream, Loader=yaml.FullLoader)

fs = 256
num_sec = 1

eeg_X = []
spec_y = []
subjects = list(data.keys())
for sbj in subjects[:1]:

    stimuli = list(metadata[sbj].keys())
    for stim in stimuli:

        response = data[sbj][stim]['response']
        interval_length = response.shape[1] // 4
        mean_eeg = torch.zeros_like(torch.empty(20, interval_length))

        for i in range(4):
            start_index = i * interval_length
            end_index = (i + 1) * interval_length
            interval = response[:, start_index:end_index]
            mean_eeg += interval
        mean_eeg /= 4

        seg = 0
        while (seg+num_sec)*fs < mean_eeg.shape[1]:
            eeg_tensor = mean_eeg[:, seg*fs:(seg+num_sec)*fs]

            stimulus = data[sbj][stim]['stimulus']
            sfreq = metadata[sbj][stim]['wav_info']['sfreq']

            # placed here due to mel-spec requiring sfreq
            transform = T.Spectrogram()

            # audio to mono
            ch1 = stimulus[0, :]
            ch2 = stimulus[1, :]
            mix = ch1 + ch2
            mix = mix[seg*sfreq:(seg+num_sec)*sfreq]

            # generating spectrogram (ideally mel-spec)
            spec_tensor = transform(torch.tensor(mix).float().unsqueeze(0)).to(device)
            eeg_X.append(eeg_tensor)
            spec_y.append(spec_tensor)

            seg += 1
    print(f'subject {sbj} data gathered')

subject 0001 data gathered


In [6]:
print(f'{len(eeg_X)} samples gathered')

n_channels, n_times = eeg_X[0].shape
n_freqs, n_bins = spec_y[0][0].shape
latent_x, latent_y, latent_z = audio_z[0][0].shape

print(f'{n_channels} channels per eeg recording')
print(f'{n_times} time steps per eeg recording')
print(f'{n_freqs} frequencies per spectrogram image')
print(f'{n_bins} time steps per spectrogram image')
print(f'{latent_x} latent x dimension')
print(f'{latent_y} latent y dimension')
print(f'{latent_z} latent z dimension')

180 samples gathered
20 channels per eeg recording
256 time steps per eeg recording
201 frequencies per spectrogram image
221 time steps per spectrogram image
16 latent x dimension
50 latent y dimension
55 latent z dimension


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define the Triplet Loss function
class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(TripletLoss, self).__init__()
        self.margin = margin
        self.loss_fn = nn.TripletMarginLoss(margin=self.margin, p=2)

    def forward(self, anchor, positive, negative):
        return self.loss_fn(anchor, positive, negative)

# Define the feature extraction network (example structure)
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(64 * 32 * 32, 128)  # Adjust according to input size

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = x.view(x.size(0), -1)  # Flatten
        x = torch.relu(self.fc1(x))
        return x

feature_extractor = FeatureExtractor()
triplet_loss = TripletLoss()

optimizer = optim.Adam(feature_extractor.parameters(), lr=1e-4)

for epoch in range(num_epochs):
    for batch in dataloader:
        anchors, positives, negatives = batch

        optimizer.zero_grad()
        anchor_features = feature_extractor(anchors)
        positive_features = feature_extractor(positives)
        negative_features = feature_extractor(negatives)

        loss = triplet_loss(anchor_features, positive_features, negative_features)
        loss.backward()
        optimizer.step()

In [ ]:
class Generator(nn.Module):
    def __init__(self, n_class=10, res=128):
        super(Generator, self).__init__()
        filters = [1024, 512, 256, 128, 64, 32]
        strides = [4, 2, 2, 2, 2, 2]
        self.cnn_depth = len(filters)

        self.cond_embedding = nn.Embedding(num_embeddings=n_class, embedding_dim=50)
        self.cond_dense = nn.Linear(50, 201 * 221 * 1)
        self.cond_reshape = nn.Unflatten(1, (1, 201, 221))

        self.conv = nn.ModuleList([
            nn.utils.spectral_norm(nn.ConvTranspose2d(in_channels=filters[idx-1] if idx > 0 else 100,
                                                      out_channels=filters[idx],
                                                      kernel_size=3,
                                                      stride=strides[idx],
                                                      padding=1,
                                                      bias=False))
            for idx in range(self.cnn_depth)
        ])
        self.act = nn.ModuleList([nn.LeakyReLU() for _ in range(self.cnn_depth)])
        self.bnorm = nn.ModuleList([nn.BatchNorm2d(num_features=filters[i]) for i in range(self.cnn_depth)])
        self.last_conv = nn.utils.spectral_norm(nn.Conv2d(in_channels=filters[-1],
                                                          out_channels=3,
                                                          kernel_size=3,
                                                          stride=1,
                                                          padding=1,
                                                          bias=False))

    def forward(self, X):
        X = X.unsqueeze(1).unsqueeze(1)
        for idx in range(self.cnn_depth):
            X = self.act[idx](self.conv[idx](X))
            X = self.bnorm[idx](X)
        X = self.last_conv(X)
        return torch.tanh(X)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, n_class=10, res=128):
        super(Discriminator, self).__init__()
        filters = [64, 128, 256, 512, 1024, 1]
        strides = [2, 2, 2, 2, 1, 1]
        self.cnn_depth = len(filters)

        self.cond_embedding = nn.Embedding(num_embeddings=n_class, embedding_dim=50)
        self.cond_dense = nn.Linear(50, res * res)
        self.cond_reshape = nn.Unflatten(1, (1, res, res))

        self.cnn_conv = nn.ModuleList([
            nn.Conv2d(in_channels=3 if i == 0 else filters[i-1],
                      out_channels=filters[i],
                      kernel_size=3,
                      stride=strides[i],
                      padding=1,
                      bias=False)
            for i in range(self.cnn_depth)
        ])
        self.cnn_bnorm = nn.ModuleList([nn.BatchNorm2d(num_features=filters[i]) for i in range(self.cnn_depth)])
        self.cnn_act = nn.ModuleList([nn.LeakyReLU(0.2) for _ in range(self.cnn_depth)])
        self.flat = nn.Flatten()
        self.disc_out = nn.Linear(in_features=filters[-1]*res//(2**len(strides))*res//(2**len(strides)), out_features=1)

    def forward(self, x, C):
        C = C.unsqueeze(1).unsqueeze(1)
        C = C.expand(-1, x.size(2), x.size(3), -1)
        x = torch.cat([x, C], dim=1)
        for i in range(self.cnn_depth):
            x = self.cnn_act[i](self.cnn_conv[i](x))
            x = self.cnn_bnorm[i](x)
        x = self.flat(x)
        x = self.disc_out(x)
        return x

In [ ]:
class DCGAN(nn.Module):
    def __init__(self):
        super(DCGAN, self).__init__()
        self.gen = Generator()
        self.disc = Discriminator()

In [ ]:
def dist_train_step(model, model_gopt, model_copt, X, C, latent_dim=96, batch_size=64):
    noise_vector = torch.rand((batch_size, latent_dim), device=X.device) * 2 - 1
    noise_vector_2 = torch.rand((batch_size, latent_dim), device=X.device) * 2 - 1
    noise_vector = torch.cat([noise_vector, C], dim=-1)
    noise_vector_2 = torch.cat([noise_vector_2, C], dim=-1)

    def train_step_disc():
        model.disc.train()
        model.gen.eval()
        model_copt.zero_grad()

        fake_img = model.gen(noise_vector)
        D_real, X_recon = model.disc(X, C)
        D_fake, _ = model.disc(fake_img, C)

        c_loss = disc_hinge(D_real, D_fake) # Define disc_hinge similarly in PyTorch

        c_loss.backward()
        model_copt.step()
        return c_loss.item()

    def train_step_gen():
        model.gen.train()
        model.disc.eval()
        model_gopt.zero_grad()

        fake_img_o = model.gen(noise_vector)
        fake_img_2_o = model.gen(noise_vector_2)
        fake_img = diff_augment(fake_img_o) # Define diff_augment similarly in PyTorch
        fake_img_2 = diff_augment(fake_img_2_o)

        D_fake, _ = model.disc(fake_img, C)
        D_fake_2, _ = model.disc(fake_img_2, C)

        g_loss = gen_hinge(D_fake) + gen_hinge(D_fake_2) # Define gen_hinge similarly in PyTorch

        mode_loss = torch.mean(torch.abs(fake_img_2_o - fake_img_o)) / torch.mean(torch.abs(noise_vector_2 - noise_vector))
        mode_loss = 1.0 / (mode_loss + 1e-5)
        g_loss += 1.0 * mode_loss

        g_loss.backward()
        model_gopt.step()
        return g_loss.item()

    discriminator_loss = train_step_disc()
    generator_loss = train_step_gen()
    return generator_loss, discriminator_loss